# First steps: three kinds of cell

An M9 notebook has three kinds of cell, and every one of them is a
whole module.

- A cell that begins `STATEFUL DEFINITION MODULE Name` is a **state
  cell**: its variables are computed once, when the cell runs, and
  every later cell may read them by name (`Name.xs`).
- A cell that holds only an **expression** shows its value, the way
  Python shows a cell's last expression.
- A cell that begins `MODULE Name` is a **program**: it is checked,
  compiled and run, and what it prints -- and any figure it writes --
  appears under it.

Run the cells in order.

In [ ]:
STATEFUL DEFINITION MODULE Squares ;

VAR xs, ys : SLICE OF F64 ;

END Squares.

IMPLEMENTATION MODULE Squares ;

IMPORT Io ;

VAR i : I64 ;

BEGIN
  xs := NEW (F64, 10) ;
  ys := NEW (F64, 10) ;
  FOR i := 0 TO 9 DO
    xs[i] := F64 (i + 1) ;
    ys[i] := xs[i] * xs[i]
  END
EXCEPT
| ValueRange :
    Io.ErrLine ('a conversion failed')
END Squares.

A state cell prints nothing of its own. Ask for a value by writing
its name -- the module is imported for you:

In [ ]:
Squares.ys

In [ ]:
Stats.Mean (Squares.ys)

In [ ]:
Arrays.NanSum (Squares.ys)

A program does anything a compiled M9 program does. This one prints a
table; `Fmt` formats the numbers, and every exception a call can raise
is handled, because the checker proves `RAISES` complete.

In [ ]:
MODULE Table ;

IMPORT Fmt ;
IMPORT Io ;
IMPORT Squares ;

VAR i : I64 ;

BEGIN
  Io.WriteLine ('    x      x^2') ;
  FOR i := 0 TO LEN (Squares.xs) - 1 DO
    Io.WriteLine (Fmt.FixedPad (Squares.xs[i], 5, 1) + '  '
                  + Fmt.FixedPad (Squares.ys[i], 7, 1))
  END
EXCEPT
| ValueRange :
    Io.ErrLine ('formatting failed')
| IndexError :
    Io.ErrLine ('out of range')
END Table.

A figure is a file: any SVG or PNG a program writes next to the
notebook is shown under the cell.

In [ ]:
MODULE Figure ;

IMPORT Io ;
IMPORT Plot ;
IMPORT Squares ;

BEGIN
  Plot.ClearFigure () ;
  Plot.AddLine (Squares.xs, Squares.ys, 0, 'x squared') ;
  Io.WriteFile ('squares.svg', Plot.Render ('the first ten squares', 'x', 'x^2'))
EXCEPT
| ValueRange :
    Io.ErrLine ('rendering failed')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
END Figure.

And a cell the checker refuses is not run at all. The message carries
the cell's own line and column; here a literal that does not fit the
variable's width.

In [ ]:
MODULE Narrow ;

VAR x : I32 ;

BEGIN
  x := 40000000000
END Narrow.